# Q6. Directed functional connectivity from population spikes

Run `q6_glm.py` and then `q6_glm_stats.R` before running this notebook. All plots are written to `results/q6/glm` by default. Set `Q6_OUTPUT_DIR` to read and plot an alternative run.

The primary GLM uses 10 ms bins and 100 ms of strictly past short history. The score is the held-out improvement in bits per target spike from including the source's short history, conditional on the target's history, the third area's short history, and the target's slower baseline. Points are mouse means over unit draws matched across areas. Error bars are marginal 95% t intervals across mice. The four sensitivity settings vary one time parameter at a time.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if 'Q6_OUTPUT_DIR' in os.environ:
    output = Path(os.environ['Q6_OUTPUT_DIR'])
else:
    repo = next(path for path in (Path.cwd(), *Path.cwd().parents)
                if (path / 'code/questions/q6_glm.py').is_file())
    output = repo / 'results/q6/glm'
mice = pd.read_csv(output / 'q6_glm_mouse_scores.csv')
means = pd.read_csv(output / 'q6_glm_stats_means.csv')
mice['direction'] = mice['source'].astype(str) + '->' + mice['target'].astype(str)
candidates = ['1->2', '3->2', '3->1']
reverse = ['2->1', '2->3', '1->3']
colors = dict(zip(candidates, ['#1B6CA8', '#A24A12', '#6B4B91']))
label = lambda direction: direction.replace('->', ' → ')
display(means.loc[means['primary']].reset_index(drop=True))

In [ ]:
primary = mice.loc[mice['primary']].pivot(index='mouse_id', columns='direction', values='bits_per_spike')
primary_means = means.loc[means['primary']].set_index('direction')
fig, axes = plt.subplots(1, 2, figsize=(9, 4.4), sharey=True, layout='constrained')
for ax, directions, title in zip(axes, [candidates, reverse], ['Q6 candidates', 'Reverse directions']):
    positions = np.arange(len(directions))
    ax.plot(positions, primary[directions].to_numpy().T, color='0.55', alpha=0.35, lw=0.8, zorder=1)
    for x, direction in enumerate(directions):
        row = primary_means.loc[direction]
        ax.scatter(np.full(len(primary), x), primary[direction], s=12, color='0.4', alpha=0.45, zorder=2)
        color = colors.get(direction, '#343434')
        ax.errorbar(x + 0.12, row.estimate,
                    yerr=[[row.estimate - row.ci95_low], [row.ci95_high - row.estimate]],
                    fmt='o', lw=2, capsize=4, color=color, zorder=3)
    ax.axhline(0, color='0.6', lw=0.8, linestyle='--')
    ax.set_xticks(positions, [label(d) for d in directions])
    ax.set(xlim=(-0.3, 2.4), xlabel='Source → target', title=title)
axes[0].set_ylabel('Held-out gain (bits per target spike)')
fig.suptitle('Directed connectivity · 10 ms bins, 100 ms history')
fig.savefig(output / 'q6_glm_primary.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
candidate_means = means.loc[means['candidate']]
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), sharey=True, layout='constrained')
panels = [(candidate_means.query('history_ms == 100'), 'bin_ms', 'Bin width (ms)', '100 ms history'),
          (candidate_means.query('bin_ms == 10'), 'history_ms', 'History window (ms)', '10 ms bins')]
for ax, (data, column, xlabel, title) in zip(axes, panels):
    settings = sorted(data[column].unique())
    for i, direction in enumerate(candidates):
        rows = data.loc[data['direction'] == direction].sort_values(column)
        x = np.array([settings.index(value) for value in rows[column]]) + (i - 1) * 0.14
        ax.errorbar(x, rows.estimate, yerr=[rows.estimate - rows.ci95_low, rows.ci95_high - rows.estimate],
                    fmt='o-', capsize=3, color=colors[direction], label=label(direction))
    primary_idx = settings.index(10 if column == 'bin_ms' else 100)
    ax.axvspan(primary_idx - 0.24, primary_idx + 0.24, color='0.93', zorder=0)
    ax.axhline(0, color='0.6', lw=0.8, linestyle='--')
    ax.set_xticks(range(len(settings)), settings)
    ax.set(xlabel=xlabel, title=title)
axes[0].set_ylabel('Mean held-out gain (bits per target spike)')
axes[1].legend(frameon=False, title='Source → target', loc='upper left', bbox_to_anchor=(1.01, 1))
fig.suptitle('Sensitivity to bin width and history window')
fig.savefig(output / 'q6_glm_sensitivity.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
diagnostic = mice.loc[mice['primary'] & mice['candidate']].dropna(subset=['shifted_bits_per_spike'])
if not diagnostic.empty:
    fig, ax = plt.subplots(figsize=(5.3, 4.6), layout='constrained')
    for direction, rows in diagnostic.groupby('direction'):
        ax.scatter(rows['shifted_bits_per_spike'], rows['bits_per_spike'],
                   alpha=0.75, s=30, color=colors[direction], label=label(direction))
    gain_range = diagnostic[['shifted_bits_per_spike', 'bits_per_spike']].to_numpy()
    lo, hi = min(gain_range.min(), 0), max(gain_range.max(), 0)
    ax.plot([lo, hi], [lo, hi], ls='--', lw=1, color='0.5', zorder=0)
    ax.set(xlabel='Shifted source (bits per target spike)', ylabel='Aligned source (bits per target spike)',
           title='Local circular-shift diagnostic', aspect='equal')
    ax.legend(frameon=False, title='Source → target')
    fig.savefig(output / 'q6_glm_shift_diagnostic.png', dpi=180, bbox_inches='tight')
    plt.show()